# 02 — FoodVision AI | EfficientNet-B0 Training, Fine-tuning & Evaluation

Cleaned-up model notebook consolidated from your two supplied EfficientNet notebooks. **Data audit and split verification live separately** in `00_food270_eda_and_split_audit.ipynb`.

The notebook retains both stages (frozen head, then fine-tuning) and your existing checkpoint paths. The **augmentation experiment has been excluded** so cells execute logically. Existing weights can be restored without re-running long training cells. If starting from scratch, first train the frozen head using your original `02_efficientnet_b0(2).ipynb` or follow the optional training cell below. 

*Do not delete the existing experiment checkpoints or split CSVs.*

## 1. Setup

In [ ]:
from pathlib import Path
import csv
import json
import io
import random

import numpy as np
import tensorflow as tf
from PIL import Image

# Change this only if your images are stored elsewhere.
DATASET_ROOT = Path(r"C:\deep_learning_projects\archive\images")
BATCH_SIZE = 16
SEED = 42
IMAGE_SIZE = (224, 224)

# Works when the notebook is opened from the project root or ml/notebooks/.
current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents)
     if ((p / "ml/data/splits/train.csv").exists() or (p / "ml/data/splits/train.csv").exists())),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate ml/data/splits/train.csv. Run the notebook inside the foodvision-ai project.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_ROOT}")

SPLITS_DIR = PROJECT_ROOT / "ml/data/splits"
if not (SPLITS_DIR / "train.csv").exists():
    SPLITS_DIR = PROJECT_ROOT / "ml/data/splits"
with (SPLITS_DIR / "class_to_index.json").open("r", encoding="utf-8") as f:
    CLASS_TO_INDEX = json.load(f)
NUM_CLASSES = len(CLASS_TO_INDEX)

tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass  # GPU may already have been initialized in this kernel.

print("Project:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("GPU:", gpus)
print("Number of food classes:", NUM_CLASSES)

## 2. Load the same training and validation images

In [ ]:
def read_split(filename):
    paths, labels = [], []
    with (SPLITS_DIR / filename).open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            relative = Path(*row["relative_path"].replace("\\", "/").split("/"))
            paths.append(str(DATASET_ROOT / relative))
            labels.append(int(row["class_index"]))
    return paths, labels


def decode_webp(encoded):
    # WebP images are decoded with Pillow; JPEG/PNG/BMP use TensorFlow.
    if hasattr(encoded, "item"):
        encoded = encoded.item()
    with Image.open(io.BytesIO(encoded)) as img:
        return np.asarray(img.convert("RGB"), dtype=np.uint8)


def read_image(path, label):
    raw = tf.io.read_file(path)
    is_webp = tf.strings.regex_full_match(tf.strings.lower(path), r".*\.webp")
    image = tf.cond(
        is_webp,
        lambda: tf.numpy_function(decode_webp, [raw], tf.uint8),
        lambda: tf.io.decode_image(raw, channels=3, expand_animations=False),
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    # Keep raw 0–255 pixels. CNN rescales inside the model;
    # EfficientNetB0 includes its own preprocessing.
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, tf.cast(label, tf.int32)


def make_dataset(filename, training=False):
    paths, labels = read_split(filename)
    if training:
        paired = list(zip(paths, labels))
        random.Random(SEED).shuffle(paired)
        paths, labels = zip(*paired)

    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        ds = ds.shuffle(min(2048, len(paths)), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(read_image, num_parallel_calls=2)
    ds = ds.batch(BATCH_SIZE).prefetch(1)
    return ds, len(paths)


train_ds, train_count = make_dataset("train.csv", training=True)
val_ds, val_count = make_dataset("val.csv", training=False)
print(f"Training: {train_count:,} images | Validation: {val_count:,} images")
print("One batch:", next(iter(train_ds))[0].shape)

## 3. Build pretrained EfficientNet-B0

The network uses pretrained ImageNet features; only the final food classifier trains initially. This is **transfer learning**, unlike the CNN trained from scratch.

In [ ]:
from tensorflow.keras import layers, Model

# Keras downloads the pretrained ImageNet weights the FIRST time.
# An internet connection is required for that initial download.
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3), name="input_image")
# Do NOT add Rescaling(1/255): EfficientNetB0 already preprocesses its inputs.
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = Model(inputs, outputs, name="foodvision_efficientnet_b0")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)
model.summary()
print("Trainable layers in feature extractor:", sum(int(l.trainable) for l in base_model.layers))

## 4. Restore your existing pretrained-head checkpoint

This is **not** training from scratch. Run setup, dataset, and model-build cells above first; do not repeat the old 10-epoch frozen-head training run.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'ml/experiments/efficientnet_b0'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HEAD_CHECKPOINT = OUTPUT_DIR / 'best_weights.h5'
FINE_TUNE_CHECKPOINT = OUTPUT_DIR / 'best_finetuned_weights.h5'

if HEAD_CHECKPOINT.exists():
    model.load_weights(str(HEAD_CHECKPOINT))
    print('Restored existing frozen-head checkpoint:', HEAD_CHECKPOINT)
else:
    print('No frozen-head checkpoint found. Run next OPTIONAL cell to train the head.')

### Optional: train frozen head (only if checkpoint is missing)

Skip this cell if `best_weights.h5` already exists. This is the first stage; it trains the Dense classification head while keeping EfficientNet frozen.

In [ ]:
if not HEAD_CHECKPOINT.exists():
    model.fit(
        train_ds, validation_data=val_ds, epochs=10,
        callbacks=[tf.keras.callbacks.ModelCheckpoint(
            str(HEAD_CHECKPOINT), monitor='val_accuracy', mode='max',
            save_best_only=True, save_weights_only=True
        )], verbose=1,
    )
    model.load_weights(str(HEAD_CHECKPOINT))
else:
    print('Skipping training: frozen-head checkpoint already exists.')

## 5. Prepare fine-tuning

Unfreeze only the last 20 backbone layers. Keep all BatchNormalization layers frozen, and use a **smaller** learning rate. The backbone was originally called with `training=False`, which is appropriate for its BatchNormalization layers.

In [ ]:
# `base_model` was created in Section 3, so no model-name search is needed.
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)
print("Trainable backbone layers:", sum(int(l.trainable) for l in base_model.layers))
print("Total trainable parameters:", sum(int(tf.keras.backend.count_params(w)) for w in model.trainable_weights))

## 6. Fine-tune and save the best weights

`EPOCHS` is a ceiling: EarlyStopping may stop earlier, but is not guaranteed to. Keep your original `best_weights.h5` unchanged.

In [ ]:
EPOCHS = 50  # Maximum fine-tuning epochs; increase later only if justified.

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINE_TUNE_CHECKPOINT),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(OUTPUT_DIR / "finetuning_log.csv")),
]

history = None
if not FINE_TUNE_CHECKPOINT.is_file():
    history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
print("Fine-tuned checkpoint:", FINE_TUNE_CHECKPOINT)

## 7. Evaluate best fine-tuned checkpoint on validation data

Select by validation accuracy. Keep the test split untouched until choosing the final model.

In [ ]:
import matplotlib.pyplot as plt

if not FINE_TUNE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"No fine-tuned checkpoint: {FINE_TUNE_CHECKPOINT}")
model.load_weights(str(FINE_TUNE_CHECKPOINT))
metrics = model.evaluate(val_ds, return_dict=True, verbose=1)
print("Best fine-tuned validation metrics:", metrics)

report = {
    "model": "efficientnet_b0_finetuned",
    "classes": NUM_CLASSES,
    "train_images": train_count,
    "validation_images": val_count,
    "validation": {k: float(v) for k, v in metrics.items()},
    "checkpoint": str(FINE_TUNE_CHECKPOINT),
}
(OUTPUT_DIR / "finetuned_metrics.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "class_to_index.json").write_text(
    json.dumps(CLASS_TO_INDEX, indent=2), encoding="utf-8"
)
if history is not None:
    plt.plot(history.history["accuracy"], label="Train accuracy")
if history is not None:
    plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("EfficientNet-B0 fine-tuning")
plt.xlabel("Fine-tuning epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Compare frozen-head and fine-tuned results

The original head checkpoint is preserved in `best_weights.h5`. The original validation metrics file exists only if you ran the previous notebook's evaluation cell.

In [ ]:
original = OUTPUT_DIR / "metrics.json"
finetuned = OUTPUT_DIR / "finetuned_metrics.json"
for label, path in [("Original frozen-head", original), ("Fine-tuned", finetuned)]:
    if path.is_file():
        report = json.loads(path.read_text(encoding="utf-8"))
        m = report["validation"]
        print(f"{label}: Top-1 {m['accuracy']:.2%} | Top-5 {m['top5_accuracy']:.2%} | loss {m['loss']:.4f}")
    else:
        print(f"{label}: no saved metrics file ({path.name})")

In [ ]:
import pandas as pd
# Load the untouched test split
test_df = pd.read_csv(
    SPLITS_DIR / "test.csv"
)

print("Test images:", len(test_df))

# Load the untouched test dataset
test_ds, test_count = make_dataset("test.csv", training=False)

# Restore the best fine-tuned model
model.load_weights(
    str(OUTPUT_DIR / "best_finetuned_weights.h5")
)

# Evaluate on test dataset
test_results = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)

print("\nFinal Test Results")
print(f"Top-1 Accuracy: {test_results['accuracy']:.2%}")
print(f"Top-5 Accuracy: {test_results['top5_accuracy']:.2%}")
print(f"Test Loss: {test_results['loss']:.4f}")


## Deployment notes

- The selected original model is `ml/experiments/efficientnet_b0/best_finetuned_weights.h5` (your demonstrated test Top-1 80.40%, Top-5 94.08%).
- Inference must build **the same backbone trainable/frozen layer configuration** as the saved checkpoint and keep pixel values 0–255 (EfficientNet preprocesses internally).
- `backend/app/services/inference.py` currently reads the mapping from `ml/data/splits/class_to_index.json`; if the reports folder is moved, update this path to `ml/data/splits/class_to_index.json` and test load again.
- The augmentation experiment checkpoint is stored separately in `ml/experiments/efficientnet_b0_augmented/` and is **not** part of this clean model notebook.
- A 270-class trained checkpoint can be too large for normal GitHub hosting. Prefer a release asset or Git LFS; exclude raw images and secrets. Verify GitHub size policies before uploading.